In [1]:
!pip install gdown


In [2]:
import gdown

KT1_ID = "1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw"
KT1_OUT = "/content/EdNet-KT1.zip"

gdown.download(f"https://drive.google.com/uc?id={KT1_ID}", KT1_OUT, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw
From (redirected): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw&confirm=t&uuid=81238ae6-a7a8-4c78-b29c-f23ffdcf3109
To: /content/EdNet-KT1.zip
100%|██████████| 1.20G/1.20G [00:21<00:00, 54.8MB/s]


'/content/EdNet-KT1.zip'

In [3]:
KT2_ID = "1qQQshbzyULW5RjMi7u-IhUr2dT_6uMXA"
KT2_OUT = "/content/EdNet-KT2.zip"

gdown.download(f"https://drive.google.com/uc?id={KT2_ID}", KT2_OUT, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1qQQshbzyULW5RjMi7u-IhUr2dT_6uMXA
From (redirected): https://drive.google.com/uc?id=1qQQshbzyULW5RjMi7u-IhUr2dT_6uMXA&confirm=t&uuid=f35d93f1-4ea6-40ff-bf4c-cfedcd448946
To: /content/EdNet-KT2.zip
100%|██████████| 556M/556M [00:07<00:00, 78.4MB/s]


'/content/EdNet-KT2.zip'

In [4]:
import zipfile, os

WORK_DIR = "/content/ednet"
KT1_DIR = "/content/ednet/KT1/KT1"


os.makedirs(KT1_DIR, exist_ok=True)

with zipfile.ZipFile("/content/EdNet-KT1.zip", "r") as zip_ref:
    zip_ref.extractall(KT1_DIR)

print("✅ KT1 extracted")


✅ KT1 extracted


In [5]:
import os

print(os.listdir("/content/ednet/KT1"))


['KT1']


In [6]:
print(os.listdir("/content/ednet/KT1/KT1")[:10])


['KT1']


In [7]:
files = os.listdir(KT1_DIR)
print("Number of student files:", len(files))
print("Sample files:", files[:5])


Number of student files: 1
Sample files: ['KT1']


In [8]:
QUESTIONS_ID = "1EsMDGRuJna4mDwRlrc2ioPO0cbH82QbP"
QUESTIONS_PATH = "/content/questions.csv"

gdown.download(
    f"https://drive.google.com/uc?id={QUESTIONS_ID}",
    QUESTIONS_PATH,
    quiet=False
)


Downloading...
From: https://drive.google.com/uc?id=1EsMDGRuJna4mDwRlrc2ioPO0cbH82QbP
To: /content/questions.csv
100%|██████████| 647k/647k [00:00<00:00, 136MB/s]


'/content/questions.csv'

In [9]:
import zipfile, os

CONTENT_ZIP_PATH = "/content/questions.csv"   # this is actually the ZIP
CONTENT_DIR = "/content/ednet/content"

os.makedirs(CONTENT_DIR, exist_ok=True)

with zipfile.ZipFile(CONTENT_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(CONTENT_DIR)

print("✅ Content ZIP extracted")


✅ Content ZIP extracted


In [10]:
import pandas as pd

questions = pd.read_excel("/content/questions.csv")  # yes, path stays same
questions.head()


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439


In [11]:
print(questions.columns.tolist())


['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']


In [12]:
questions = questions[[
    "question_id",
    "correct_answer",
    "tags"
]]


In [13]:
import pandas as pd
import os
from glob import glob

# Path to extracted KT1 CSV files
KT1_DATA_DIR = "/content/ednet/KT1/KT1"

# Load a small subset for now
student_files = glob(f"{KT1_DATA_DIR}/*.csv")[:50]

dfs = []
for file in student_files:
    df = pd.read_csv(file)
    df["user_id"] = os.path.basename(file).replace(".csv", "")
    dfs.append(df)

kt1 = pd.concat(dfs, ignore_index=True)

print("kt1 created with shape:", kt1.shape)


ValueError: No objects to concatenate

In [14]:
import os

print("ednet exists:", os.path.exists("/content/ednet"))
print("KT1 exists:", os.path.exists("/content/ednet/KT1"))

if os.path.exists("/content/ednet/KT1"):
    print("Inside /content/ednet/KT1:", os.listdir("/content/ednet/KT1"))


ednet exists: True
KT1 exists: True
Inside /content/ednet/KT1: ['KT1']


In [15]:
print(os.listdir("/content/ednet/KT1/KT1")[:10])


['KT1']


In [ ]:
kt1.head()


In [ ]:
kt1 = kt1.merge(
    questions,
    how="left",
    on="question_id"
)


In [ ]:
kt1["correct"] = (
    kt1["user_answer"] == kt1["correct_answer"]
).astype(int)


In [ ]:
def parse_tags(tag):
    if pd.isna(tag):
        return []
    return [int(t) for t in str(tag).split(";")]

kt1["skills"] = kt1["tags"].apply(parse_tags)


In [ ]:
kt1 = kt1.sort_values(
    by=["user_id", "timestamp"]
).reset_index(drop=True)


In [ ]:
student_sequences = {}

for uid, df in kt1.groupby("user_id"):
    student_sequences[uid] = {
        "questions": df["question_id"].tolist(),
        "correct": df["correct"].tolist(),
        "skills": df["skills"].tolist()
    }


In [ ]:
import numpy as np

MAX_SEQ_LEN = 100

def pad(seq):
    return seq[-MAX_SEQ_LEN:] if len(seq) >= MAX_SEQ_LEN else \
           [0] * (MAX_SEQ_LEN - len(seq)) + seq

X_q, X_a = [], []

for s in student_sequences.values():
    X_q.append(pad(s["questions"]))
    X_a.append(pad(s["correct"]))

X_q = np.array(X_q)
X_a = np.array(X_a)


In [ ]:
np.save("/content/X_questions.npy", X_q)
np.save("/content/X_answers.npy", X_a)


In [16]:
# =========================
# EdNet KT1 Preprocessing
# =========================

# Install dependency
!pip install gdown

# Imports
import gdown
import os
import zipfile
import pandas as pd
import numpy as np
from glob import glob

# -------------------------
# 1. Download EdNet-KT1
# -------------------------
KT1_ID = "1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw"
KT1_ZIP = "/content/EdNet-KT1.zip"

gdown.download(f"https://drive.google.com/uc?id={KT1_ID}", KT1_ZIP, quiet=False)

# -------------------------
# 2. Extract KT1 properly
# -------------------------
BASE_DIR = "/content/ednet"
KT1_EXTRACT_DIR = f"{BASE_DIR}/KT1"

os.makedirs(KT1_EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(KT1_ZIP, "r") as zip_ref:
    zip_ref.extractall(KT1_EXTRACT_DIR)

KT1_DATA_DIR = f"{KT1_EXTRACT_DIR}/KT1"
print("KT1 sample files:", os.listdir(KT1_DATA_DIR)[:5])

# -------------------------
# 3. Download & load questions.xlsx
# -------------------------
QUESTIONS_ID = "1EsMDGRuJna4mDwRlrc2ioPO0cbH82QbP"
QUESTIONS_PATH = "/content/questions.xlsx"

gdown.download(
    f"https://drive.google.com/uc?id={QUESTIONS_ID}",
    QUESTIONS_PATH,
    quiet=False
)

questions = pd.read_excel(QUESTIONS_PATH)
questions = questions[["question_id", "correct_answer", "tags"]]
print("Questions columns:", questions.columns.tolist())

# -------------------------
# 4. Load KT1 student logs
# -------------------------
student_files = glob(f"{KT1_DATA_DIR}/*.csv")[:50]  # subset for MVP
print("Number of students loaded:", len(student_files))

dfs = []
for file in student_files:
    df = pd.read_csv(file)
    df["user_id"] = os.path.basename(file).replace(".csv", "")
    dfs.append(df)

kt1 = pd.concat(dfs, ignore_index=True)
print("KT1 dataframe shape:", kt1.shape)

# -------------------------
# 5. Merge + correctness label
# -------------------------
kt1 = kt1.merge(questions, how="left", on="question_id")

kt1["correct"] = (
    kt1["user_answer"] == kt1["correct_answer"]
).astype(int)

# -------------------------
# 6. Parse skill tags
# -------------------------
def parse_tags(tag):
    if pd.isna(tag):
        return []
    return [int(t) for t in str(tag).split(";")]

kt1["skills"] = kt1["tags"].apply(parse_tags)

# -------------------------
# 7. Sort by time
# -------------------------
kt1 = kt1.sort_values(
    by=["user_id", "timestamp"]
).reset_index(drop=True)

# -------------------------
# 8. Build student sequences
# -------------------------
student_sequences = {}

for uid, df in kt1.groupby("user_id"):
    student_sequences[uid] = {
        "questions": df["question_id"].tolist(),
        "correct": df["correct"].tolist()
    }

# -------------------------
# 9. Pad & save tensors
# -------------------------
MAX_SEQ_LEN = 100

def pad(seq):
    return seq[-MAX_SEQ_LEN:] if len(seq) >= MAX_SEQ_LEN else \
           [0] * (MAX_SEQ_LEN - len(seq)) + seq

X_q, X_a = [], []

for s in student_sequences.values():
    X_q.append(pad(s["questions"]))
    X_a.append(pad(s["correct"]))

X_q = np.array(X_q)
X_a = np.array(X_a)

np.save("/content/X_questions.npy", X_q)
np.save("/content/X_answers.npy", X_a)

print("✅ Preprocessing completed successfully")
print("X_questions shape:", X_q.shape)
print("X_answers shape:", X_a.shape)


Downloading...
From (original): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw
From (redirected): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw&confirm=t&uuid=08dd7f26-13e7-4108-a74d-ad4a373a4076
To: /content/EdNet-KT1.zip
100%|██████████| 1.20G/1.20G [00:26<00:00, 45.3MB/s]


KT1 sample files: ['u540392.csv', 'u526560.csv', 'u283834.csv', 'u831313.csv', 'u406555.csv']


Downloading...
From: https://drive.google.com/uc?id=1EsMDGRuJna4mDwRlrc2ioPO0cbH82QbP
To: /content/questions.xlsx
100%|██████████| 647k/647k [00:00<00:00, 123MB/s]


Questions columns: ['question_id', 'correct_answer', 'tags']
Number of students loaded: 50
KT1 dataframe shape: (3913, 6)
✅ Preprocessing completed successfully
X_questions shape: (50, 100)
X_answers shape: (50, 100)


In [17]:
from google.colab import files

# Download preprocessed KT tensors
files.download("/content/X_questions.npy")
files.download("/content/X_answers.npy")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>